# **SpaceX Falcon 9 — Exploratory Data Analysis with Visualization**

In this lab we explore the wrangled SpaceX dataset and engineer the features used by the
predictive models. We visualize the relationships between launch attributes (flight number,
payload mass, launch site, orbit) and the landing outcome (`Class`: 1 = successful landing,
0 = unsuccessful), then one-hot encode the categorical features and export `dataset_part_3.csv`.


In [ ]:
!pip install pandas
!pip install numpy
!pip install matplotlib
!pip install seaborn

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Load the wrangled dataset (output of the data-wrangling stage)

In [ ]:
URL = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/dataset_part_2.csv"
df = pd.read_csv(URL)
df.head(5)

## TASK 1: Flight Number vs. Payload Mass
Each point is a launch, colored by landing outcome. We look for how payload mass and the
accumulation of flight experience relate to success.

In [ ]:
sns.catplot(y="PayloadMass", x="FlightNumber", hue="Class", data=df, aspect=5)
plt.xlabel("Flight Number", fontsize=20)
plt.ylabel("Payload Mass (kg)", fontsize=20)
plt.title("Flight Number vs. Payload Mass by landing outcome")
plt.show()

## TASK 2: Flight Number vs. Launch Site
We check whether certain launch sites became more successful as flight number increased.

In [ ]:
sns.catplot(x="FlightNumber", y="LaunchSite", hue="Class", data=df, aspect=2)
plt.xlabel("Flight Number", fontsize=15)
plt.ylabel("Launch Site", fontsize=15)
plt.title("Flight Number vs. Launch Site by landing outcome")
plt.show()

## TASK 3: Payload Mass vs. Launch Site
We examine how payload mass varies across launch sites and how that relates to success.

In [ ]:
sns.catplot(x="PayloadMass", y="LaunchSite", hue="Class", data=df, aspect=2)
plt.xlabel("Payload Mass (kg)", fontsize=15)
plt.ylabel("Launch Site", fontsize=15)
plt.title("Payload Mass vs. Launch Site by landing outcome")
plt.show()

## TASK 4: Success rate of each orbit type
We compute the mean of `Class` per orbit (the landing success rate) and rank orbits.

In [ ]:
orbit_success = df.groupby("Orbit")["Class"].mean().reset_index().sort_values("Class", ascending=False)
plt.figure(figsize=(10, 5))
sns.barplot(x="Orbit", y="Class", data=orbit_success)
plt.xlabel("Orbit", fontsize=15)
plt.ylabel("Success rate", fontsize=15)
plt.title("Landing success rate by orbit type")
plt.show()
orbit_success

## TASK 5: Flight Number vs. Orbit type
We look for trends in success across orbit types as flight number increases.

In [ ]:
sns.catplot(x="FlightNumber", y="Orbit", hue="Class", data=df, aspect=2)
plt.xlabel("Flight Number", fontsize=15)
plt.ylabel("Orbit", fontsize=15)
plt.title("Flight Number vs. Orbit type by landing outcome")
plt.show()

## TASK 6: Payload Mass vs. Orbit type
We examine how payload mass interacts with orbit type and outcome.

In [ ]:
sns.catplot(x="PayloadMass", y="Orbit", hue="Class", data=df, aspect=2)
plt.xlabel("Payload Mass (kg)", fontsize=15)
plt.ylabel("Orbit", fontsize=15)
plt.title("Payload Mass vs. Orbit type by landing outcome")
plt.show()

## TASK 7: Yearly launch success trend
We extract the launch year from the date and plot the average success rate over time.

In [ ]:
def Extract_year(dframe):
    year = []
    for d in dframe["Date"]:
        year.append(d.split("-")[0])
    return year

df_year = df.copy()
df_year["Year"] = Extract_year(df_year)
yearly = df_year.groupby("Year")["Class"].mean().reset_index()
yearly["Year"] = yearly["Year"].astype(int)

plt.figure(figsize=(10, 5))
sns.lineplot(x="Year", y="Class", data=yearly, marker="o")
plt.xlabel("Year", fontsize=15)
plt.ylabel("Average success rate", fontsize=15)
plt.title("Falcon 9 landing success rate by year")
plt.show()
yearly

## Features Engineering
We select the columns used for modeling and one-hot encode the categorical variables
(`Orbit`, `LaunchSite`, `LandingPad`, `Serial`). All columns are cast to `float64` so the
result is a fully numeric feature matrix.

In [ ]:
features = df[['FlightNumber', 'PayloadMass', 'Orbit', 'LaunchSite', 'Flights', 'GridFins',
               'Reused', 'Legs', 'LandingPad', 'Block', 'ReusedCount', 'Serial']]
features.head()

In [ ]:
# One-hot encode the categorical columns
features_one_hot = pd.get_dummies(features, columns=['Orbit', 'LaunchSite', 'LandingPad', 'Serial'])
features_one_hot.head()

In [ ]:
# Cast the entire feature matrix to float64
features_one_hot = features_one_hot.astype('float64')
features_one_hot.head()

## Export the feature matrix for the predictive-modeling stage

In [ ]:
features_one_hot.to_csv('dataset_part_3.csv', index=False)
print('Saved dataset_part_3.csv ->', features_one_hot.shape)